## Part 1 - Setup

In [1]:
# Uncomment these if running for the first time

# !pip install playwright pandas
# !playwright install chromium

In [2]:
from playwright.async_api import async_playwright, Error as PlaywrightError
import pandas as pd
import re
from pathlib import Path

In [ ]:
CITIES_CSV = "../data/reference/pakistan_cities_search_urls.csv"

cities_df = pd.read_csv(CITIES_CSV)

HEADLESS = False

SLOW_MO = 0

output_dir = Path("../data/raw")
output_dir.mkdir(exist_ok=True, parents=True)

processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True, parents=True)

print(f"Loaded {len(cities_df)} cities to scrape")
cities_df.head()

In [4]:
playwright = await async_playwright().start()

browser = await playwright.chromium.launch(
    headless=HEADLESS,
    slow_mo=SLOW_MO
)

context = await browser.new_context(
    viewport={"width": 1600, "height": 900},
    locale="en-US",
    java_script_enabled=True
)

page = await context.new_page()

print("Browser launched")

Browser launched


## Part 2 - Helper Functions

In [5]:
async def dismiss_signin_popup(page):
    try:
        dismiss_btn = page.locator('button[aria-label="Dismiss sign in information."]')
        await dismiss_btn.wait_for(state="visible", timeout=5000)
        await dismiss_btn.click()
        return True
    except:
        return False

In [6]:
async def accept_cookies(page):
    possible_buttons = [
        "Accept",
        "Accept all",
        "I agree",
        "Got it",
        "Allow all"
    ]

    for text in possible_buttons:
        try:
            btn = page.get_by_role("button", name=text)

            if await btn.count() > 0:
                await btn.first.click(timeout=2000)
                return True

        except:
            pass

    return False

In [7]:
async def scroll_to_bottom(page, times=3, wait_ms=1000):
    for i in range(times):
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await page.wait_for_timeout(wait_ms)

In [8]:
async def load_all_results(page):

    previous_count = 0

    while True:

        try:

            cards = page.locator('[data-testid="property-card"]')

            current_count = await cards.count()

        except PlaywrightError as e:

            print(f"Browser/page closed unexpectedly ({e}). Stopping.")

            break

        print(f"Loaded properties : {current_count}")

        if current_count == previous_count:
            print("No new properties detected.")

        previous_count = current_count

        load_more = page.get_by_role(
            "button",
            name=re.compile("Load more", re.IGNORECASE)
        )

        if await load_more.count() == 0:
            print("No Load More button found.")
            break

        try:

            await load_more.first.scroll_into_view_if_needed()

            await page.wait_for_timeout(400)

            await load_more.first.click(timeout=5000)

            print("Clicked Load More")

        except PlaywrightError as e:

            print(f"Browser/page closed unexpectedly ({e}). Stopping.")

            break

        except Exception as e:

            print(f"Could not click button: {e}")

            break

        try:

            await page.wait_for_function(
                f"""
                () => document.querySelectorAll(
                '[data-testid="property-card"]'
                ).length > {current_count}
                """,
                timeout=20000
            )

        except PlaywrightError as e:

            print(f"Browser/page closed unexpectedly ({e}). Stopping.")

            break

        except:

            print("No additional properties loaded.")

            break

        await page.wait_for_timeout(400)

    print("Finished Loading")

In [9]:
async def extract_properties(page):
    properties = await page.eval_on_selector_all(
        '[data-testid="property-card"]',
        """
        (cards) => {

            function text(root, selector) {
                const el = root.querySelector(selector);
                return el ? el.innerText.trim() : null;
            }

            function attr(root, selector, name) {
                const el = root.querySelector(selector);
                return el ? el.getAttribute(name) : null;
            }

            function findByText(root, regex) {
                const walker = document.createTreeWalker(root, NodeFilter.SHOW_ELEMENT);
                let node;
                while (node = walker.nextNode()) {
                    if (node.children.length === 0 && regex.test(node.innerText || "")) {
                        return node.innerText.trim();
                    }
                }
                return null;
            }

            function bedType(root) {
                const units = root.querySelector('[data-testid="recommended-units"]');
                return units ? findByText(units, /\\bbeds?\\b/i) : null;
            }

            return cards.map((card) => ({
                "property_name": text(card, '[data-testid="title"]'),
                "property_url": attr(card, "a", "href"),
                "Price_pkr": text(card, '[data-testid="price-and-discounted-price"]'),
                "review_score": text(card, '[data-testid="review-score"] > :nth-child(2)'),
                "review_count": text(card, '[data-testid="review-score"] > :nth-child(3) > :nth-child(2)'),
                "address": text(card, '[data-testid="address-link"]'),
                "distance_from_center_km": text(card, '[data-testid="distance"]'),
                "property_type": text(card, '[data-testid="recommended-units"] > div > div > h4'),
                "bed_type": bedType(card),
                "breakfast_included": findByText(card, /Breakfast/i),
                "free_cancellation": findByText(card, /Free cancellation/i),
                "reserve_without_payment": findByText(card, /No prepayment/i),
                "image": attr(card, "img", "src"),
                "stars": card.querySelectorAll("svg").length,
            }));
        }
        """
    )

    return properties

In [10]:
def clean_dataframe(df):
    df = df.drop_duplicates(subset=["property_url"])

    df["Price_pkr"] = pd.to_numeric(
        df["Price_pkr"].astype(str).str.replace(r"[^\d]", "", regex=True),
        errors="coerce"
    )

    df["review_count"] = pd.to_numeric(
        df["review_count"].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce"
    )

    df["distance_from_center_km"] = pd.to_numeric(
        df["distance_from_center_km"].astype(str).str.extract(r"([\d.]+)")[0],
        errors="coerce"
    )

    df["breakfast_included"] = df["breakfast_included"].notna().astype(int)
    df["free_cancellation"] = df["free_cancellation"].notna().astype(int)
    df["reserve_without_payment"] = df["reserve_without_payment"].notna().astype(int)

    return df

## Part 3 - Loop Through All Cities

In [11]:
all_dataframes = []

for i, row in cities_df.iterrows():

    city = row["city"]
    url = row["url"]

    print("=" * 60)
    print(f"[{i + 1}/{len(cities_df)}] Scraping {city}")
    print("=" * 60)

    try:
        await page.goto(url)
        await page.wait_for_load_state("networkidle")

        await dismiss_signin_popup(page)
        await accept_cookies(page)
        await scroll_to_bottom(page)

        await load_all_results(page)

        properties = await extract_properties(page)

        if not properties:
            print(f"No properties found for {city}, skipping.")
            continue

        city_df = pd.DataFrame(properties)
        city_df = clean_dataframe(city_df)
        city_df["city"] = city

        city_filename = output_dir / f"{city.replace(' ', '_')}.csv"
        city_df.to_csv(city_filename, index=False, encoding="utf-8-sig")

        print(f"Saved {len(city_df)} properties for {city} -> {city_filename}")

        all_dataframes.append(city_df)

    except PlaywrightError as e:
        print(f"Browser/page closed unexpectedly while scraping {city} ({e}). Stopping loop.")
        break

    except Exception as e:
        print(f"Error scraping {city}: {e}. Skipping to next city.")
        continue

    await page.wait_for_timeout(3000)

print("\nAll cities processed.")

[1/79] Scraping Islamabad
Loaded properties : 74
Clicked Load More
Loaded properties : 98
Clicked Load More
Loaded properties : 122
Clicked Load More
Loaded properties : 144
Clicked Load More
Loaded properties : 168
Clicked Load More
Loaded properties : 192
Clicked Load More
Loaded properties : 217
Clicked Load More
Loaded properties : 239
Clicked Load More
Loaded properties : 263
Clicked Load More
Loaded properties : 286
Clicked Load More
Loaded properties : 309
Clicked Load More
Loaded properties : 334
Clicked Load More
Loaded properties : 359
Clicked Load More
Loaded properties : 383
Clicked Load More
Loaded properties : 406
Clicked Load More
Loaded properties : 431
Clicked Load More
Loaded properties : 453
Clicked Load More
Loaded properties : 478
Clicked Load More
Loaded properties : 500
Clicked Load More
Loaded properties : 522
Clicked Load More
Loaded properties : 547
Clicked Load More
Loaded properties : 569
Clicked Load More
Loaded properties : 594
Clicked Load More
Loaded pro

## Part 4 - Combine & Save

In [ ]:
if all_dataframes:
    combined_df = pd.concat(all_dataframes, ignore_index=True)

    combined_filename = processed_dir / "all_cities_combined.csv"

    combined_df.to_csv(combined_filename, index=False, encoding="utf-8-sig")

    print("=" * 60)
    print(f"Combined {len(combined_df)} properties from {len(all_dataframes)} cities")
    print(f"Saved to {combined_filename}")
    print("=" * 60)
else:
    print("No data collected.")

In [13]:
await browser.close()

await playwright.stop()

print("Browser closed")

Browser closed
